# `select` — Reference

`select` picks columns by name, slice, dtype, or substring. Negative names (`'-col'`, `'~col'`), negative slices (`'-start:end'`), and `exclude=` drop columns while preserving relative order. Regex is the `regex=` keyword — a positional string that is not an exact name is an error, not a silent regex. `everything()` pulls in "the rest" after pinning specific columns first.

| Argument type | Example | Selects |
|---|---|---|
| List | `['a','b']` | exact named columns |
| String (name) | `'species'` | exact column |
| Negative string | `'-species'`, `'~island'` | exclude named column |
| Negative slice | `'-bill_length_mm:body_mass_g'` | exclude contiguous range |
| `exclude=` | `exclude='species'`, `exclude=['a','b']` | exclude specific column(s) |
| String (slice) | `'bill_length_mm:body_mass_g'` | contiguous range |
| `everything()` | after explicit cols | remaining columns |
| callable | `lambda c: c.startswith('b')` | columns passing test |
| `regex=` | `regex='mm$'` | names matching regex |
| `dtype=` | `dtype='numeric'` | by dtype |
| `exclude_dtype=` | `exclude_dtype='number'` | excluding dtype |
| `contains=` | `contains='mm'` | substring in name |
| `startswith=` | `startswith='bill'` | name prefix |
| `endswith=` | `endswith='mm'` | name suffix |

---

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
titanic = pt.sample_data['titanic']
tips = pt.sample_data['tips']

penguins.pt.cols(ascending=None)  # see column order

['species',
 'island',
 'bill_length_mm',
 'bill_depth_mm',
 'flipper_length_mm',
 'body_mass_g',
 'sex']

## Names, `contains`, and `regex`
Criteria are a union (OR). Prefer `contains=` for substrings and `regex=` for patterns. Multiple values in a list are also OR.

In [2]:
# All columns that contain 'bill' OR 'mass', plus species
(penguins
 .pt.select('species', contains=['bill', 'mass'])
 .head()
)

,species,bill_length_mm,bill_depth_mm,body_mass_g
0,Adelie,39.1,18.7,3750.0
1,Adelie,39.5,17.4,3800.0
2,Adelie,40.3,18.0,3250.0
3,Adelie,NaN,NaN,NaN
4,Adelie,36.7,19.3,3450.0


In [3]:
# Column order rule: explicit list preserves YOUR order; contains/regex follow original DataFrame order
# island, sex, species are listed explicitly so they come first in that exact order,
# then contains='mm' adds remaining mm-cols in original df order
(penguins
 .pt.select('island', 'sex', 'species', contains='mm')
 .head()
)

,island,sex,species,bill_length_mm,bill_depth_mm,flipper_length_mm
0,Torgersen,Male,Adelie,39.1,18.7,181.0
1,Torgersen,Female,Adelie,39.5,17.4,186.0
2,Torgersen,Female,Adelie,40.3,18.0,195.0
3,Torgersen,NaN,Adelie,NaN,NaN,NaN
4,Torgersen,Female,Adelie,36.7,19.3,193.0


In [4]:
# Columns ending in 'mm' — anchored regex
(penguins
 .pt.select(regex='mm$')
 .head()
)

,bill_length_mm,bill_depth_mm,flipper_length_mm
0,39.1,18.7,181.0
1,39.5,17.4,186.0
2,40.3,18.0,195.0
3,NaN,NaN,NaN
4,36.7,19.3,193.0


In [5]:
# Combining contains= with a slice in one call (union)
# contains='mm' picks the mm-cols; 'island:body_mass_g' slice picks everything between those two cols
(penguins
 .pt.select('island:body_mass_g', contains='mm')
 .head()
)

,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,Torgersen,39.1,18.7,181.0,3750.0
1,Torgersen,39.5,17.4,186.0,3800.0
2,Torgersen,40.3,18.0,195.0,3250.0
3,Torgersen,NaN,NaN,NaN,NaN
4,Torgersen,36.7,19.3,193.0,3450.0


## Slice notation — contiguous column range
`'start_col:end_col'` selects all columns from start to end (inclusive), in DataFrame order.

In [6]:
# Select from 'bill_length_mm' through 'body_mass_g' (all measurement cols)
(penguins
 .pt.select('bill_length_mm:body_mass_g')
 .head()
)

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,39.1,18.7,181.0,3750.0
1,39.5,17.4,186.0,3800.0
2,40.3,18.0,195.0,3250.0
3,NaN,NaN,NaN,NaN
4,36.7,19.3,193.0,3450.0


In [7]:
# Advanced regex: columns containing 'mm' but NOT 'flipper'
# this is regex for selecting all columns containing 'mm' but not 'flipper'.
# just because it can be done, does not mean it should be done.
# use more readable ways like list comprehension — this will ensure your co-workers
# don't stab you in the middle of the night
(penguins
 .pt.select(regex=r'^(?=.*mm)(?!.*flipper).*')  # regex sorcery
 .head()
)

# More readable alternative:
cols_with_mm_not_flipper = [c for c in penguins.columns if 'mm' in c and 'flipper' not in c]
(
    penguins
    .pt.select(cols_with_mm_not_flipper)
    .head()
)


,bill_length_mm,bill_depth_mm
0,39.1,18.7
1,39.5,17.4
2,40.3,18.0
3,NaN,NaN
4,36.7,19.3


## `everything()` — pin key columns first, then the rest
Use `everything()` to move specific columns to the front without dropping any.

In [8]:
# Move 'body_mass_g' and 'species' to front, keep all other columns after
(penguins
 .pt.select('body_mass_g', 'species', pt.everything())
 .head()
)

,body_mass_g,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,sex
0,3750.0,Adelie,Torgersen,39.1,18.7,181.0,Male
1,3800.0,Adelie,Torgersen,39.5,17.4,186.0,Female
2,3250.0,Adelie,Torgersen,40.3,18.0,195.0,Female
3,NaN,Adelie,Torgersen,NaN,NaN,NaN,NaN
4,3450.0,Adelie,Torgersen,36.7,19.3,193.0,Female


## Callable — select columns with a function

In [9]:
# Lambda: columns whose name has more than 6 characters
(penguins
 .pt.select(lambda c: len(c) > 6)
 .head()
)

,species,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,Adelie,39.1,18.7,181.0,3750.0
1,Adelie,39.5,17.4,186.0,3800.0
2,Adelie,40.3,18.0,195.0,3250.0
3,Adelie,NaN,NaN,NaN,NaN
4,Adelie,36.7,19.3,193.0,3450.0


## `dtype`, `exclude_dtype`, `contains`, `startswith`, `endswith`, `regex`

In [10]:
# All numeric columns only
(penguins
 .pt.select(dtype='numeric')
 .head()
)

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,39.1,18.7,181.0,3750.0
1,39.5,17.4,186.0,3800.0
2,40.3,18.0,195.0,3250.0
3,NaN,NaN,NaN,NaN
4,36.7,19.3,193.0,3450.0


In [11]:
# Drop numeric columns — keep only categorical/string columns
(penguins
 .pt.select(exclude_dtype='number')
 .head()
)

,species,island,sex
0,Adelie,Torgersen,Male
1,Adelie,Torgersen,Female
2,Adelie,Torgersen,Female
3,Adelie,Torgersen,NaN
4,Adelie,Torgersen,Female


In [12]:
# exclude_dtype also accepts a list — exclude multiple dtype families at once
(penguins
 .pt.select(exclude_dtype=['datetime', 'object'])
 .head()
)

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,39.1,18.7,181.0,3750.0
1,39.5,17.4,186.0,3800.0
2,40.3,18.0,195.0,3250.0
3,NaN,NaN,NaN,NaN
4,36.7,19.3,193.0,3450.0


In [13]:
# Titanic: columns starting with 'p' or 'a', plus all numeric; behaviour here is | and not &
(titanic
 .pt.select(startswith=['p', 'a'], dtype='numeric')
 .head()
)

,survived,pclass,age,sibsp,parch,fare,adult_male,alive,alone
0,0,3,22.0,1,0,7.2500,True,no,False
1,1,1,38.0,1,0,71.2833,False,yes,False
2,1,3,26.0,0,0,7.9250,False,yes,True
3,1,1,35.0,1,0,53.1000,False,yes,False
4,0,3,35.0,0,0,8.0500,True,no,True


In [14]:
# endswith — select columns whose name ends with a given suffix
(penguins
 .pt.select(endswith='mm')  # same result as .select(regex='mm$') but more readable
 .head()
)

,bill_length_mm,bill_depth_mm,flipper_length_mm
0,39.1,18.7,181.0
1,39.5,17.4,186.0
2,40.3,18.0,195.0
3,NaN,NaN,NaN
4,36.7,19.3,193.0


In [15]:
# Columns containing 'bill' or 'flipper' in name, used in a full chain ;  behaviour here is | and not &
(
    penguins
    .pt.qry(species=['Adelie', 'Gentoo'])
    .pt.select('species', 'island', contains=['bill', 'flipper'])
    .pt.agg(['species', 'island'], a=['mean', 'n'])
)


,species,island,n,bill_length_mm,bill_depth_mm,flipper_length_mm
0,Adelie,Biscoe,44,38.975000,18.370455,188.795455
1,Adelie,Dream,56,38.501786,18.251786,189.732143
2,Adelie,Torgersen,52,38.950980,18.429412,191.196078
3,Gentoo,Biscoe,124,47.504878,14.982114,217.186992


## Negative selection & exclusion — `-col`, `~col`, negative slices, and `exclude=`
Drop or exclude columns by prefixing names with `-` or `~`, passing a negative slice (`'-start:end'`), or using the `exclude=` parameter. Remaining columns preserve their original relative DataFrame order without needing a separate drop function.

In [16]:
# Drop single column by negating with '-' or '~'
penguins.pt.select('-species').head(3)

,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Torgersen,40.3,18.0,195.0,3250.0,Female


In [17]:
# Drop multiple columns
penguins.pt.select('-species', '-island', '-sex').head(3)

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,39.1,18.7,181.0,3750.0
1,39.5,17.4,186.0,3800.0
2,40.3,18.0,195.0,3250.0


In [18]:
# Exclude a contiguous slice of columns
penguins.pt.select('-bill_length_mm:body_mass_g').head(3)

,species,island,sex
0,Adelie,Torgersen,Male
1,Adelie,Torgersen,Female
2,Adelie,Torgersen,Female


In [19]:
# Exclude columns using the exclude= keyword
penguins.pt.select(exclude=['species', 'island']).head(3)

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,39.1,18.7,181.0,3750.0,Male
1,39.5,17.4,186.0,3800.0,Female
2,40.3,18.0,195.0,3250.0,Female


In [20]:
# Combine positive selection with negative exclusion in a chain
(penguins
 .pt.select(dtype='numeric')
 .pt.select('-bill_depth_mm')
 .head(3)
)

,bill_length_mm,flipper_length_mm,body_mass_g
0,39.1,181.0,3750.0
1,39.5,186.0,3800.0
2,40.3,195.0,3250.0
